# Bias, Variance & Regularization

## Linear and Logistic Regression

The two workhorses of classical supervised learning — **linear regression** and **logistic regression** — share a common structure: a linear map from inputs $\mathbf{x} \in \mathbb{R}^d$ to predictions, parameterized by weights $\mathbf{w} \in \mathbb{R}^d$ and a scalar bias $b$. They differ only in what happens to that linear output and how the loss is defined.

**Linear regression.** For regression problems, the prediction is the raw linear output $\hat{y} = \mathbf{w}^\top \mathbf{x} + b.$ Given a dataset $\mathcal{D} = (\mathbf{x}_i, y_i)_{i=1}^N$ with $y_i \in \mathbb{R}$, the squared-error loss is:

$$\mathcal{L}(\mathbf{w}) = \frac{1}{N} \sum_{i=1}^N (\mathbf{w}^\top \mathbf{x}_i + b - y_i)^2 = \frac{1}{N} \lVert \mathbf{X}\mathbf{w} - \mathbf{y} \rVert^2,$$

where $\mathbf{X} \in \mathbb{R}^{N \times d}$ absorbs the bias into an augmented feature vector. This is a quadratic in $\mathbf{w}$, and setting $\nabla_{\mathbf{w}} \mathcal{L} = 0$ gives the **normal equations** with the unique minimizer:

$$\boxed{\hat{\mathbf{w}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}.}$$

This closed-form solution is exact and $O(Nd^2 + d^3)$ to compute. For large $d$ the cubic inversion dominates, making gradient descent more practical despite requiring more iterations.

**Probabilistic derivation.** Assume labels follow $y_i = \mathbf{w}^\top \mathbf{x}_i + \epsilon_i$ with $\epsilon_i \sim \mathcal{N}(0, \sigma^2).$ Then $p(y_i \mid \mathbf{x}_i; \mathbf{w}) = \mathcal{N}(\mathbf{w}^\top \mathbf{x}_i, \sigma^2).$ Maximum likelihood estimation — maximizing $\prod_i p(y_i \mid \mathbf{x}_i; \mathbf{w})$ — is equivalent to minimizing the squared-error loss. The Gaussian noise assumption is what makes MSE the canonical loss for regression.

**Logistic regression.** For binary classification with $y_i \in \{0, 1\}$, we model

$$p(y = 1 \mid \mathbf{x}; \mathbf{w}) = \sigma(\mathbf{w}^\top \mathbf{x} + b), \quad \sigma(z) = \frac{1}{1 + e^{-z}}.$$

Maximizing the log-likelihood of the Bernoulli model over $\mathcal{D}$ gives:

$$\hat{\mathbf{w}} = \underset{\mathbf{w}}{\operatorname{argmax}} \sum_{i=1}^N \left[ y_i \log \sigma(\mathbf{w}^\top \mathbf{x}_i) + (1 - y_i) \log (1 - \sigma(\mathbf{w}^\top \mathbf{x}_i)) \right].$$

The negative of this objective is the **binary cross-entropy loss**. Unlike linear regression, no closed form exists — the logistic function introduces a nonlinearity that prevents the normal equations from reducing to a linear system. We must use gradient descent or Newton's method.

**Regularization as MAP estimation.** Adding an $L_2$ penalty to the loss corresponds exactly to placing a zero-mean Gaussian prior on the weights and performing maximum a posteriori (MAP) estimation:

$$\hat{\mathbf{w}}_{\text{MAP}} = \underset{\mathbf{w}}{\operatorname{argmin}} \left[ \mathcal{L}(\mathbf{w}) + \frac{\lambda}{2} \lVert \mathbf{w} \rVert^2 \right], \quad \mathbf{w} \sim \mathcal{N}(0, \lambda^{-1} \mathbf{I}).$$

An $L_1$ penalty corresponds to a Laplace prior. This probabilistic interpretation justifies regularization not as an ad hoc trick, but as the natural consequence of encoding beliefs about the scale of parameters before seeing data. We develop this connection rigorously in NB03.

Imports and plot setup for the notebook:

In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Fitting linear and logistic regression on toy data using scikit-learn, then comparing with the closed-form normal equations:

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.datasets import make_classification

# --- linear regression: closed-form vs. sklearn ---
np.random.seed(RANDOM_SEED)
N, d = 100, 1
X = np.random.randn(N, d)
w_true = np.array([2.5])
y = X @ w_true + 0.5 * np.random.randn(N)          # <1>

# normal equations
X_aug = np.hstack([X, np.ones((N, 1))])             # <2>
w_ols = np.linalg.lstsq(X_aug, y, rcond=None)[0]   # <3>

# sklearn
lr = LinearRegression().fit(X, y)

print(f"Normal equations: w={w_ols[0]:.4f}, b={w_ols[1]:.4f}")
print(f"sklearn:          w={lr.coef_[0]:.4f}, b={lr.intercept_:.4f}")

1. Generating $y = \mathbf{w}^\top \mathbf{x} + \epsilon$ with $\epsilon \sim \mathcal{N}(0, 0.25).$
2. Absorbing the bias into the feature matrix by appending a column of ones.
3. `np.linalg.lstsq` is numerically preferred over explicit inversion via $(\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$ when $\mathbf{X}^\top \mathbf{X}$ is ill-conditioned.

## Bias-Variance Tradeoff

We decompose the expected test error of a learned model into three additive components that illuminate the fundamental tension in supervised learning. For a squared-error regression problem, the expected prediction error at a test point $\mathbf{x}_0$ is:

$$\mathbb{E}_{\mathcal{D}, \epsilon}\left[(y_0 - \hat{f}(\mathbf{x}_0))^2\right] = \underbrace{\left(\mathbb{E}_\mathcal{D}[\hat{f}(\mathbf{x}_0)] - f(\mathbf{x}_0)\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}_\mathcal{D}\left[\left(\hat{f}(\mathbf{x}_0) - \mathbb{E}_\mathcal{D}[\hat{f}(\mathbf{x}_0)]\right)^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible noise}}.$$

The expectation over $\mathcal{D}$ is over the randomness in the training set — conceptually, we imagine retraining the model on many different draws of size $N$ from the same data-generating process. The **bias** is the systematic error of the model class: how far the average prediction is from the truth. The **variance** captures how much predictions fluctuate across training draws. The **irreducible noise** $\sigma^2$ is the inherent label noise that no model can eliminate.

**Derivation.** Let $f_0 = f(\mathbf{x}_0)$ and $\bar{f} = \mathbb{E}_\mathcal{D}[\hat{f}(\mathbf{x}_0)].$ Then $y_0 = f_0 + \epsilon$ with $\mathbb{E}[\epsilon] = 0$, $\mathbb{E}[\epsilon^2] = \sigma^2.$ Expanding the squared error and using $\mathbb{E}[(y_0 - \hat{f})^2] = \mathbb{E}[(f_0 + \epsilon - \hat{f})^2]$ and independence between $\epsilon$ and $\hat{f}$:

$$\begin{aligned}
\mathbb{E}[(y_0 - \hat{f})^2] &= \mathbb{E}[(f_0 - \hat{f})^2] + \sigma^2 \\[0.75em]
&= \mathbb{E}[(f_0 - \bar{f} + \bar{f} - \hat{f})^2] + \sigma^2 \\[0.75em]
&= (f_0 - \bar{f})^2 + \mathbb{E}[(\hat{f} - \bar{f})^2] + 2(f_0 - \bar{f})\underbrace{\mathbb{E}[\hat{f} - \bar{f}]}_{=0} + \sigma^2.
\end{aligned}$$

The cross term vanishes by definition of $\bar{f}$, yielding the decomposition.

**Tradeoff.** As model complexity increases, bias decreases (the model class becomes expressive enough to represent the truth) but variance increases (the model overfits idiosyncrasies of the training set). The [optimal complexity]{.mark} balances the two, producing the classical U-shaped test error curve. The figure below illustrates this using polynomial regression on noisy sinusoidal data across 50 random splits — the shaded band shows the standard deviation across splits, making the growing instability of high-degree fits clearly visible.

In [ ]:
#| label: fig-bias-variance-tradeoff
#| fig-cap: "Bias-variance tradeoff as a function of polynomial degree on noisy sinusoidal data. Mean train and test MSE (solid lines) and ±1 std band (shaded) over 50 random 80/20 splits. The test error curve is U-shaped; the train error decreases monotonically. The dashed vertical line marks the degree with minimum mean test error."
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
rng = np.random.RandomState(0)

# --- data generating process ---
N_total = 150
X_all = rng.uniform(0, 1, N_total)
Y_all = np.sin(2 * np.pi * X_all) + rng.normal(0, 0.3, N_total)
x_true = np.linspace(0, 1, 300)
y_true = np.sin(2 * np.pi * x_true)

degrees = np.arange(1, 16)
N_splits = 50
train_mses = np.zeros((len(degrees), N_splits))
test_mses  = np.zeros((len(degrees), N_splits))

for s in range(N_splits):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all.reshape(-1, 1), Y_all, test_size=0.2, random_state=s
    )
    for i, deg in enumerate(degrees):
        model = Pipeline([
            ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
            ("lr",   LinearRegression()),
        ])
        model.fit(X_tr, y_tr)
        train_mses[i, s] = mean_squared_error(y_tr, model.predict(X_tr))
        test_mses[i, s]  = mean_squared_error(y_te, model.predict(X_te))

mean_train = train_mses.mean(axis=1)
std_train  = train_mses.std(axis=1)
mean_test  = test_mses.mean(axis=1)
std_test   = test_mses.std(axis=1)
best_deg   = degrees[mean_test.argmin()]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# --- left panel: U-curve ---
ax = axes[0]
ax.plot(degrees, mean_train, color="C0", linewidth=1.5, label="train MSE")
ax.fill_between(degrees, mean_train - std_train, mean_train + std_train,
                color="C0", alpha=0.25)
ax.plot(degrees, mean_test, color="C1", linewidth=1.5, label="test MSE")
ax.fill_between(degrees, mean_test - std_test, mean_test + std_test,
                color="C1", alpha=0.25)
ax.axvline(best_deg, color="gray", linestyle="dashed", lw=1.0,
           label=f"best degree = {best_deg}")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("MSE")
ax.set_ylim(bottom=0)
ax.legend(fontsize=8)
ax.grid(linestyle="dotted", alpha=0.6)
ax.set_title("bias-variance tradeoff")

# --- right panel: fitted curves at three representative degrees ---
ax2 = axes[1]
X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
    X_all.reshape(-1, 1), Y_all, test_size=0.2, random_state=0
)
ax2.scatter(X_tr_full, y_tr_full, s=8, alpha=0.5, color="gray", label="train points")
ax2.plot(x_true, y_true, color="black", lw=1.5, label="true function")

showcase = [(1, "C0", "degree 1 (underfit)"),
            (best_deg, "C2", f"degree {best_deg} (optimal)"),
            (14, "C3", "degree 14 (overfit)")]
for deg, color, lbl in showcase:
    m = Pipeline([
        ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
        ("lr",   LinearRegression()),
    ])
    m.fit(X_tr_full, y_tr_full)
    ax2.plot(x_true, np.clip(m.predict(x_true.reshape(-1, 1)), -3, 3),
             color=color, lw=1.5, label=lbl)

ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_ylim(-2.5, 2.5)
ax2.legend(fontsize=7, loc="upper right")
ax2.grid(linestyle="dotted", alpha=0.6)
ax2.set_title("fitted curves")

fig.tight_layout()
plt.show()

## L1 and L2 Regularization

Regularization constrains the hypothesis class by penalizing the magnitude of the weights. We add a penalty term to the empirical loss:

$$\mathcal{L}_{\text{reg}}(\mathbf{w}) = \mathcal{L}(\mathbf{w}) + \lambda \Omega(\mathbf{w}),$$

where $\lambda \geq 0$ controls the strength of regularization and $\Omega$ is the penalty. The two canonical choices are (1) **$L_2$ regularization** (Ridge): $\Omega(\mathbf{w}) = \frac{1}{2}\lVert \mathbf{w} \rVert_2^2$, and (2) **$L_1$ regularization** (Lasso): $\Omega(\mathbf{w}) = \lVert \mathbf{w} \rVert_1 = \sum_j |w_j|.$

**Geometric intuition.** Regularized estimation can be re-expressed as a constrained problem: minimize the loss subject to $\Omega(\mathbf{w}) \leq t$ for some budget $t$ that decreases as $\lambda$ increases. The feasible regions are a ball ($L_2$, smooth, round) and a diamond ($L_1$, with corners at the coordinate axes). The unconstrained minimizer $\hat{\mathbf{w}}_\text{OLS}$ lies somewhere in $\mathbb{R}^d$, and the regularized solution is where the loss contours first touch the feasible region. For $L_2$, that contact point is generically in the interior of a facet — all coordinates are shrunk but nonzero. For $L_1$, the loss contours frequently first touch a [corner of the diamond]{.mark}, which sits on a coordinate axis — setting that coefficient to exactly zero. This is the mechanism of **sparsity induction** in Lasso.

**Effect on coefficient shrinkage.** For linear regression with $L_2$ regularization, the solution is closed-form:

$$\hat{\mathbf{w}}_\text{Ridge} = (\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}.$$

As $\lambda \to \infty$, all coefficients are shrunk toward zero. The $\lambda \mathbf{I}$ term also ensures the matrix is invertible even when $\mathbf{X}^\top \mathbf{X}$ is singular (which occurs whenever $d > N$ or features are collinear), making Ridge a practical remedy for ill-conditioned regression.

For $L_1$, there is no closed form. The subgradient optimality condition $0 \in \partial_w \mathcal{L} + \lambda \partial |w|$ yields the **soft-thresholding** operation: small OLS coefficients with $|\hat{w}_j| < \lambda / (\mathbf{x}_j^\top \mathbf{x}_j)$ are shrunk to exactly zero. The threshold is hard; Ridge never zeroes coefficients exactly.

**Elastic Net** combines both penalties: $\Omega(\mathbf{w}) = \alpha \lVert \mathbf{w} \rVert_1 + \frac{1-\alpha}{2} \lVert \mathbf{w} \rVert_2^2,$ inheriting the sparsity of Lasso and the grouping effect of Ridge (correlated features tend to be selected together).

**Regularization paths.** The behavior of coefficients as $\lambda$ varies continuously is called the **regularization path**. For Ridge, all coefficients shrink smoothly toward zero as $\lambda \to \infty.$ For Lasso, coefficients are driven to zero at different values of $\lambda$ — the path consists of piecewise-linear segments with kinks at each zero-crossing. Scanning this path reveals which features the model finds most predictive, independent of a fixed $\lambda$.

In [ ]:
#| label: fig-regularization-path
#| fig-cap: "Regularization paths for Ridge (left) and Lasso (right) on the diabetes dataset. Each colored line is one coefficient. As regularization strength $\\lambda$ increases (moving right), Ridge shrinks all coefficients toward zero smoothly; Lasso drives them to exactly zero at different $\\lambda$ values. The dashed vertical line marks $\\lambda$ selected by 5-fold CV."
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import lasso_path, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")

X_dia, y_dia = load_diabetes(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dia)

# --- Lasso path via sklearn's efficient LARS-based solver ---
alphas_lasso, coefs_lasso, _ = lasso_path(X_scaled, y_dia, n_alphas=100)
lasso_cv = LassoCV(cv=5, n_alphas=100, random_state=0).fit(X_scaled, y_dia)
best_alpha_lasso = lasso_cv.alpha_

# --- Ridge path via manual alpha sweep ---
from sklearn.linear_model import Ridge
alphas_ridge = np.logspace(-2, 4, 100)
coefs_ridge = np.array([
    Ridge(alpha=a).fit(X_scaled, y_dia).coef_
    for a in alphas_ridge
])  # shape (100, n_features)
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5).fit(X_scaled, y_dia)
best_alpha_ridge = ridge_cv.alpha_

feature_names = load_diabetes().feature_names

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Ridge path
for j in range(coefs_ridge.shape[1]):
    ax1.plot(np.log10(alphas_ridge), coefs_ridge[:, j],
             lw=1.2, label=feature_names[j])
ax1.axvline(np.log10(best_alpha_ridge), color="gray", linestyle="dashed",
            lw=1.0, label=f"CV $\\lambda$ = {best_alpha_ridge:.2f}")
ax1.set_xlabel(r"$\log_{10}(\lambda)$")
ax1.set_ylabel("coefficient value")
ax1.set_title("Ridge path")
ax1.legend(fontsize=6, loc="upper right", ncol=2)
ax1.grid(linestyle="dotted", alpha=0.6)

# Lasso path (sklearn returns alphas in decreasing order; reverse for left-to-right plot)
log_alphas_lasso = np.log10(alphas_lasso[::-1])
coefs_lasso_r = coefs_lasso[:, ::-1]
for j in range(coefs_lasso_r.shape[0]):
    ax2.plot(log_alphas_lasso, coefs_lasso_r[j],
             lw=1.2, label=feature_names[j])
ax2.axvline(np.log10(best_alpha_lasso), color="gray", linestyle="dashed",
            lw=1.0, label=f"CV $\\lambda$ = {best_alpha_lasso:.3f}")
ax2.set_xlabel(r"$\log_{10}(\lambda)$")
ax2.set_ylabel("coefficient value")
ax2.set_title("Lasso path")
ax2.legend(fontsize=6, loc="upper left", ncol=2)
ax2.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show()

## Cross-Validation and Model Selection

The training error $\mathcal{L}_\text{train}$ is a biased, optimistic estimate of true generalization error. Any model that sees the training set during fitting is evaluated on data it has already adapted to, so $\mathcal{L}_\text{train} \leq \mathcal{L}_\text{test}$ in expectation — a gap called the **optimism of training error**. This gap grows with model complexity: a degree-14 polynomial can interpolate any 15 points, making training MSE zero while test MSE explodes.

**$k$-fold cross-validation** is the standard solution. We partition the training data into $k$ equal folds, train on $k-1$ folds, evaluate on the held-out fold, and average the $k$ error estimates. This yields a nearly unbiased estimate of test error at the cost of $k$ full training runs.

Typical choices: $k = 5$ or $k = 10.$ **Leave-one-out CV** (LOOC, $k = N$) is asymptotically unbiased but expensive for large $N$; for linear models, the LOOC score can be computed analytically in one pass using the hat matrix $\mathbf{H} = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$ via the identity:

$$\text{LOOCV} = \frac{1}{N} \sum_{i=1}^N \left(\frac{y_i - \hat{y}_i}{1 - H_{ii}}\right)^2.$$

**Stratified CV** is the default for classification: it ensures each fold has approximately the same class proportions as the full dataset, preventing folds dominated by one class from inflating variance of the CV estimate.

**The model selection problem and nested CV.** A subtler issue arises when we use CV to choose hyperparameters ($\lambda$, degree, etc.) and then report the CV score of the winning configuration. This is sometimes called **selection bias**: the chosen model is selected partly because it happened to perform well on the held-out folds, and the reported score is again optimistic. **Nested CV** resolves this by wrapping the entire model selection procedure (including hyperparameter search) in an outer CV loop:

$$\text{outer loop: generalization estimate} \supset \text{inner loop: hyperparameter selection}.$$

The outer loop provides an unbiased estimate of the generalization error of the model selection *procedure*; the inner loop selects hyperparameters on training data only, without touching the outer test fold.

:::{.callout-caution}
Never select hyperparameters — even informally, by inspecting which value "looks better" — using the held-out test set. Once a model has been evaluated on the test set, that evaluation is no longer a valid estimate of future performance. The test set must be used at most once.

:::

Demonstrating the optimism of training error by sweeping polynomial degree and comparing training error against 5-fold CV error:

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import numpy as np

rng = np.random.RandomState(0)
N = 80
X_cv = rng.uniform(0, 1, (N, 1))
y_cv = np.sin(2 * np.pi * X_cv.ravel()) + rng.normal(0, 0.3, N)

degrees = np.arange(1, 15)
train_scores, cv_scores = [], []

for deg in degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
        ("lr",   LinearRegression()),
    ])
    model.fit(X_cv, y_cv)
    train_mse = np.mean((model.predict(X_cv) - y_cv) ** 2)          # <1>
    cv_mse = -cross_val_score(                                       # <2>
        model, X_cv, y_cv, cv=5, scoring="neg_mean_squared_error"
    ).mean()
    train_scores.append(train_mse)
    cv_scores.append(cv_mse)

best = degrees[np.argmin(cv_scores)]
print(f"Best degree by 5-fold CV: {best}")
print(f"Train MSE at best degree: {train_scores[best - 1]:.4f}")
print(f"CV    MSE at best degree: {cv_scores[best - 1]:.4f}")

1. Training error is computed on the same data the model was fitted on — always an optimistic estimate.
2. `cross_val_score` returns *negative* MSE (sklearn convention for minimization); we negate it to recover positive MSE.

Demonstrating nested CV: the outer loop estimates generalization error, the inner `GridSearchCV` selects the best degree without touching the outer test fold:

In [ ]:
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score

param_grid = {"poly__degree": list(range(1, 12))}
inner_cv = KFold(n_splits=5, shuffle=True, random_state=1)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=2)

base_model = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("lr",   LinearRegression()),
])

# inner loop: hyperparameter selection
inner_search = GridSearchCV(
    base_model, param_grid, cv=inner_cv, scoring="neg_mean_squared_error"
)

# outer loop: unbiased estimate of the selection procedure's generalization error
nested_scores = cross_val_score(
    inner_search, X_cv, y_cv, cv=outer_cv, scoring="neg_mean_squared_error"
)
nested_mse = -nested_scores.mean()
print(f"Nested CV MSE: {nested_mse:.4f} ± {nested_scores.std():.4f}")

## Sklearn Pipelines

A common source of bugs in machine learning pipelines is **data leakage**: information from the test fold leaks into the fitted preprocessor, causing the CV score to be optimistic. The canonical example is standardization: if we fit a `StandardScaler` on the full dataset before splitting, the scaler's mean and variance reflect the test fold's statistics, making the test evaluation invalid.

:::{.callout-caution}
Always fit preprocessors on training data only. Fitting a scaler (or imputer, encoder, or any stateful transformer) on the concatenation of train and test folds before cross-validation is data leakage. The test fold must be statistically invisible during fitting.

:::

**`Pipeline` for leakage-free CV.** Scikit-learn's `Pipeline` chains transformers and an estimator into a single object. When `GridSearchCV` or `cross_val_score` calls `.fit()` on a pipeline, each fold's transformer is fitted only on that fold's training split — the test fold is only seen by `.transform()`. This makes leakage impossible by construction:

```
Pipeline: [StandardScaler] → [LogisticRegression]
             .fit(X_train)      .fit(X_train_scaled)
             .transform(X_test) .predict(X_test_scaled)
```

**`ColumnTransformer` for heterogeneous features.** Real tabular data mixes numeric, categorical, and sometimes ordinal columns that require different preprocessing. `ColumnTransformer` applies different transformers to different column subsets and concatenates the results:

```
ColumnTransformer:
    numeric_cols  → StandardScaler
    categorical_cols → OneHotEncoder
```

The result plugs into a `Pipeline` as a single preprocessing step. **`set_output(transform="pandas")`** (scikit-learn ≥ 1.2) makes the pipeline output a DataFrame with proper column names, which is essential for inspecting feature importance in downstream models.

**Hyperparameter search over pipeline parameters.** `GridSearchCV` and `RandomizedSearchCV` accept parameter grids keyed by `stepname__paramname`, allowing search over both preprocessor and estimator hyperparameters simultaneously. For example, `{"clf__C": [0.1, 1.0, 10.0]}` sweeps the regularization strength of a `LogisticRegression` named `clf` inside the pipeline.

Building a concrete pipeline on the breast cancer dataset — numeric features are standardized, then fed to a logistic regressor, with `GridSearchCV` tuning the regularization strength `C`:

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report
import numpy as np

X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=0
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000)),
])

param_grid = {
    "clf__C":       [0.01, 0.1, 1.0, 10.0, 100.0],  # <1>
    "clf__penalty": ["l1", "l2"],                    # <2>
    "clf__solver":  ["saga"],                         # <3>
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
search = GridSearchCV(pipeline, param_grid, cv=cv,
                      scoring="accuracy", n_jobs=-1)
search.fit(X_tr, y_tr)

print(f"Best params : {search.best_params_}")
print(f"CV accuracy : {search.best_score_:.4f}")
print(f"Test accuracy: {search.score(X_te, y_te):.4f}")

1. `clf__C` addresses the `C` parameter of the `LogisticRegression` step named `clf`. The double underscore is scikit-learn's convention for addressing nested parameters in pipelines.
2. We sweep both $L_1$ and $L_2$ penalties jointly with the regularization strength.
3. The `saga` solver supports both $L_1$ and $L_2$ penalties and scales to large datasets; `liblinear` supports only $L_1$ and $L_2$ but does not parallelize across cores.

For data with heterogeneous column types, `ColumnTransformer` handles the routing. The following example uses the titanic-style adult income dataset pattern with numeric and categorical columns:

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.datasets import fetch_openml
import numpy as np

# Titanic dataset (binary: survived vs. not)
titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = titanic.frame[["pclass", "sex", "age", "fare", "survived"]].dropna()

X_tit = df.drop(columns="survived")
y_tit = df["survived"].astype(int)

numeric_cols     = ["age", "fare"]
categorical_cols = ["pclass", "sex"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(),                     numeric_cols),   # <1>
    ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_cols),  # <2>
])

pipeline_tit = Pipeline([
    ("prep", preprocessor),
    ("clf",  LogisticRegression(C=1.0, max_iter=500)),
])

scores = cross_val_score(pipeline_tit, X_tit, y_tit, cv=5, scoring="accuracy")
print(f"Titanic 5-fold accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

1. `StandardScaler` is applied only to continuous numeric columns; fitting is restricted to training folds via the pipeline.
2. `OneHotEncoder(drop="first")` drops the first category to avoid the **dummy variable trap** (perfect multicollinearity). `sparse_output=False` keeps the output as a dense array compatible with the downstream logistic regressor.

## Double Descent

The classical U-curve predicts that increasing model complexity beyond the sweet spot raises test error. Modern overparameterized models — neural networks with millions of parameters trained on thousands of examples — routinely violate this prediction. See [Double Descent](/courses/deep-learning/18-appendix-double-descent.html) in the Deep Learning Foundations appendices for a full treatment, including reproduction of the Nakkiran et al. curves on ResNet / CIFAR-10.

---

■